# pandas — Technical Reference

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Core query pattern | Any data manipulation | §1 |
| Aggregations | Summarize rows | §2 |
| Conditional logic | Label or bucket values | §3 |
| Joins & merges | Combine DataFrames | §4 |
| Groupby & pivot | Reshape data | §5 |
| Window operations | Rank, shift, cumulative, rolling | §6 |
| Date & time | Date arithmetic, extraction | §7 |
| NULL handling | Missing values | §8 |
| Product metrics | DAU/WAU/MAU, cohort, retention | §9 |
| Performance tips | Speed and memory optimization | §10 |

---
## When to Use

| Signal | pandas method to reach for |
| :--- | :--- |
| Filter rows by condition | `.query()` or boolean indexing |
| Summarize with aggregation | `.groupby().agg()` |
| Rank within a group | `.groupby().rank()` |
| Compare to previous / next row | `.groupby().shift()` |
| Running total within a group | `.groupby().cumsum()` |
| Pivot rows to columns | `.pivot_table()` |
| Combine two DataFrames | `.merge()` |
| Find rows with no match | `merge(..., how='left', indicator=True)` |
| Label or bucket values | `np.select()` (multi-condition) or `np.where()` (binary) |
| Moving average | `.rolling(n).mean()` |
| Apply window result back to original shape | `.groupby().transform()` |

---
## §1 — Core Query Pattern

In [ ]:
import pandas as pd
import numpy as np

# SQL → pandas equivalent pipeline
# SELECT col1, AGG(col2) FROM t WHERE cond GROUP BY col1 HAVING ... ORDER BY ... LIMIT n
result = (
    df.query("condition")                           # WHERE
      .groupby("col1", as_index=False)              # GROUP BY
      .agg(agg_col=("col2", "sum"))                 # SELECT + aggregate
      .query("agg_col > 100")                       # HAVING
      .sort_values("col1", ascending=False)         # ORDER BY
      .head(10)                                     # LIMIT
)

# as_index=False keeps group keys as columns (equivalent to SELECT col1 in GROUP BY)
# Named aggregation: agg(new_col=("source_col", "function"))

**Execution order reminder:** pandas executes left to right in a chain — unlike SQL's fixed execution order. Filter with `.query()` as early as possible to reduce rows before expensive operations.

**Common mistakes:**
- Forgetting `as_index=False` in `groupby` — without it, group keys become the index, not columns
- Chaining without parentheses — Python requires wrapping the entire chain in `()` for multi-line chains
- Using `.query()` with column names that have spaces — wrap in backticks: `` .query("`column name` > 0") ``

---
## §2 — Aggregations

In [ ]:
from collections import Counter

# Basic aggregations
len(df)                                 # COUNT(*)
df["col"].count()                       # COUNT(col) — excludes NULLs
df["user_id"].nunique()                 # COUNT(DISTINCT user_id)
df["sales"].sum()                       # SUM
df["price"].mean()                      # AVG — NULLs excluded automatically
df["score"].min()
df["score"].max()

# Named aggregations in groupby
result = df.groupby("region", as_index=False).agg(
    total_sales  = ("sales",  "sum"),
    avg_price    = ("price",  "mean"),
    unique_users = ("user_id","nunique"),
    min_score    = ("score",  "min")
)

# Conditional aggregation — SUM(CASE WHEN ...) equivalent
df["ios_rev"]     = df["revenue"].where(df["platform"] == "iOS",     0)  # else 0
df["ios_rev_avg"] = df["revenue"].where(df["platform"] == "iOS")         # else NaN (auto-excluded from mean)

result = df.groupby("region", as_index=False).agg(
    ios_rev     = ("ios_rev",     "sum"),
    ios_rev_avg = ("ios_rev_avg", "mean")
)

**Common mistakes:**
- Using `.mean()` when NULLs should count as 0 — use `.sum() / len(df)` instead
- `df["col"].count()` vs `len(df)` — `.count()` excludes NaN, `len()` includes all rows
- Forgetting that `.agg()` with a list of functions returns a MultiIndex — use named aggregation syntax to avoid this

---
## §3 — Conditional Logic

In [ ]:
# np.where — binary condition (IF equivalent)
df["spender_type"] = np.where(df["spend"] > 100, "High", "Low")

# np.select — multi-condition (CASE WHEN equivalent)
conditions = [
    df["spend"] > 100,
    df["spend"].between(50, 100)
]
choices = ["High", "Medium"]
df["spender_type"] = np.select(conditions, choices, default="Low")

# pd.cut — bin continuous values into labeled ranges
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 18, 35, 60, 100],
    labels=["Under 18", "18-35", "36-60", "60+"]
)

# map — replace values using a dictionary
df["platform_label"] = df["platform_code"].map({1: "iOS", 2: "Android", 3: "Web"})

**Common mistakes:**
- Conditions in `np.select` are evaluated top to bottom — first matching condition wins; order matters
- Using Python `if/else` inside a loop instead of `np.where` / `np.select` — always use vectorized operations
- `df["col"].map(dict)` returns NaN for keys not in the dictionary — add a `.fillna()` if needed

---
## §4 — Joins & Merges

In [ ]:
# Basic merge — SQL JOIN equivalent
A.merge(B, on="id", how="inner")        # INNER JOIN
A.merge(B, on="id", how="left")         # LEFT JOIN
A.merge(B, on="id", how="right")        # RIGHT JOIN
A.merge(B, on="id", how="outer")        # FULL OUTER JOIN

# Join on multiple keys
A.merge(B, on=["user_id", "date"], how="left")

# Join on differently named columns
A.merge(B, left_on="user_id", right_on="id", how="inner")

# Handle overlapping column names
A.merge(B, on="id", how="left", suffixes=("_a", "_b"))

# Find rows in A with NO match in B — LEFT JOIN ... WHERE B.id IS NULL
merged = A.merge(B, on="id", how="left", indicator=True)
no_match = merged.loc[merged["_merge"] == "left_only"].drop(columns="_merge")

# Self join — join DataFrame to itself
employees.merge(
    employees[["id", "name"]].rename(columns={"id": "manager_id", "name": "manager_name"}),
    on="manager_id",
    how="left"
)

**Common mistakes:**
- Merging on columns with different dtypes (e.g. `int` vs `str`) — merge silently produces no matches; cast first
- Forgetting `suffixes` when both DataFrames have overlapping column names — pandas adds `_x` / `_y` by default, which is easy to confuse
- Using `.join()` instead of `.merge()` — `.join()` joins on index by default, not on a column; `.merge()` is almost always what you want

---
## §5 — Groupby & Pivot

In [ ]:
# groupby + agg — most common pattern
df.groupby("region", as_index=False).agg(
    total = ("revenue", "sum"),
    count = ("user_id", "nunique")
)

# pivot_table — rows × columns × values
# Equivalent to GROUP BY + conditional aggregation
df.pivot_table(
    index="region",                     # rows
    columns="platform",                 # columns
    values="revenue",                   # values
    aggfunc="sum",                      # aggregation
    fill_value=0                        # replace NaN with 0
).reset_index()

# crosstab — frequency table (count only)
pd.crosstab(df["region"], df["platform"])

# transform — apply aggregation but keep original shape
# Equivalent to window function without collapsing rows
df["group_total"] = df.groupby("region")["revenue"].transform("sum")
df["pct_of_group"] = df["revenue"] / df["group_total"]

**groupby + agg vs pivot_table:**
- Use `groupby + agg` when aggregating multiple columns with different functions
- Use `pivot_table` when reshaping one metric across two categorical dimensions
- Use `transform` when you need the aggregated value back on every original row

**Common mistakes:**
- `pivot_table` without `fill_value=0` — missing combinations produce NaN instead of 0
- Using `agg` when you need `transform` — `agg` collapses rows; `transform` preserves the original index
- `crosstab` only counts rows — pass `values` and `aggfunc` arguments for other aggregations

---
## §6 — Window Operations

In [ ]:
# Ranking — SQL RANK / DENSE_RANK / ROW_NUMBER equivalents
df["row_num"]   = df["score"].rank(method="first",   ascending=False).astype(int)  # ROW_NUMBER
df["rnk"]       = df["score"].rank(method="min",     ascending=False).astype(int)  # RANK
df["dense_rnk"] = df["score"].rank(method="dense",   ascending=False).astype(int)  # DENSE_RANK

# Ranking within a group — RANK() OVER (PARTITION BY ...)
df["rank_in_group"] = (
    df.groupby("region")["revenue"]
      .rank(method="dense", ascending=False)
      .astype(int)
)

# LAG / LEAD — shift within a group
df = df.sort_values(["user_id", "date"])
df["prev_value"] = df.groupby("user_id")["value"].shift(1)   # LAG(value, 1)
df["next_value"] = df.groupby("user_id")["value"].shift(-1)  # LEAD(value, 1)

# Running totals — cumulative functions
df["run_sum"] = df.groupby("user_id")["amount"].cumsum()     # SUM OVER (ROWS UNBOUNDED PRECEDING)
df["run_max"] = df.groupby("user_id")["amount"].cummax()
df["run_min"] = df.groupby("user_id")["amount"].cummin()

# Rolling window — moving average over last n rows
df["moving_avg_7"] = (
    df.groupby("user_id")["amount"]
      .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

**Rank method comparison:**

| method | Ties | Gaps | SQL equivalent |
| :--- | :--- | :--- | :--- |
| `first` | No — sequential | n/a | `ROW_NUMBER()` |
| `min` | Yes — share min rank | Yes | `RANK()` |
| `dense` | Yes — share rank | No | `DENSE_RANK()` |

**Common mistakes:**
- Forgetting to `sort_values` before `shift` — shift operates on current row order, not logical order; always sort first
- Using `groupby().cumsum()` without sorting — produces incorrect running totals
- Using `groupby().apply(lambda x: x.rolling(...))` when `groupby().transform(lambda x: x.rolling(...))` is cleaner and faster

---
## §7 — Date & Time

In [ ]:
# Parse strings to datetime
df["event_ts"] = pd.to_datetime(df["event_ts"])

# Extract components — dt accessor
df["date"]         = df["event_ts"].dt.date          # DATE()
df["year"]         = df["event_ts"].dt.year           # EXTRACT(YEAR)
df["month"]        = df["event_ts"].dt.month          # EXTRACT(MONTH)
df["day_of_week"]  = df["event_ts"].dt.dayofweek     # 0=Mon, 6=Sun
df["year_month"]   = df["event_ts"].dt.to_period("M").astype(str)  # DATE_FORMAT('%Y-%m')

# Date arithmetic
df["plus_7d"]    = df["order_date"] + pd.Timedelta(days=7)          # DATE_ADD(..., INTERVAL 7 DAY)
df["minus_1mo"]  = df["order_date"] - pd.DateOffset(months=1)       # DATE_SUB(..., INTERVAL 1 MONTH)
df["days_diff"]  = (df["end_date"] - df["start_date"]).dt.days      # DATEDIFF

# Current date
today = pd.Timestamp.today().normalize()        # CURRENT_DATE
df_recent = df.loc[df["event_date"] >= today - pd.Timedelta(days=7)]

# Unix timestamp conversion
# 10-digit = seconds, 13-digit = milliseconds
df["dt"] = pd.to_datetime(df["ts_col"], unit="s")    # seconds → datetime
df["dt"] = pd.to_datetime(df["ts_col"], unit="ms")   # milliseconds → datetime
df["date"] = pd.to_datetime(df["ts_col"], unit="s").dt.date  # seconds → date only

**Common mistakes:**
- Forgetting to call `pd.to_datetime()` before using `.dt` accessor — raises `AttributeError` if column is still a string
- Mixing `pd.Timedelta` (fixed duration) with `pd.DateOffset` (calendar-aware) — use `DateOffset` for months/years since months have variable lengths
- Using `.dt.date` (returns Python `date` objects) when you need `.dt.normalize()` (returns `Timestamp`) — the former breaks further date arithmetic

---
## §8 — NULL Handling

In [ ]:
# Check for NULLs
df.isna()                               # IS NULL
df.notna()                              # IS NOT NULL
df["col"].isna().sum()                  # count NULLs in a column

# Fill NULLs
df["salary"].fillna(0)                  # IFNULL(salary, 0)
df["contact"] = df["phone"].fillna(df["email"]).fillna("Unknown")  # COALESCE(phone, email, 'Unknown')

# Drop rows with NULLs
df.dropna()                             # drop rows with any NULL
df.dropna(subset=["user_id", "date"])   # drop rows where specific columns are NULL

# Fill forward / backward within a group
df["value"] = df.groupby("user_id")["value"].ffill()   # forward fill within group
df["value"] = df.groupby("user_id")["value"].bfill()   # backward fill within group

# NaN behavior in aggregations
df["col"].sum()     # NaN ignored
df["col"].mean()    # NaN ignored (not treated as 0)
df["col"].count()   # excludes NaN
len(df)             # includes NaN rows

**Common mistakes:**
- Using `df["col"] == None` or `df["col"] == np.nan` — always use `.isna()` or `.notna()`; equality checks on NaN always return False
- `fillna(0)` before `.mean()` when NULLs should be excluded — changes the denominator and inflates the mean
- `ffill()` without sorting first — fills in row order, not logical time order

---
## §9 — Product Metrics Patterns

In [ ]:
today = pd.Timestamp.today().normalize()

# DAU / WAU / MAU
DAU = events.loc[events["event_date"] >= today - pd.Timedelta(days=1),  "user_id"].nunique()
WAU = events.loc[events["event_date"] >= today - pd.Timedelta(days=7),  "user_id"].nunique()
MAU = events.loc[events["event_date"] >= today - pd.Timedelta(days=30), "user_id"].nunique()

# Cohort retention skeleton
cohort = (
    events.groupby("user_id", as_index=False)["event_date"]
          .min()
          .rename(columns={"event_date": "cohort_date"})
)
activity = events.merge(cohort, on="user_id")
activity["week_num"] = ((activity["event_date"] - activity["cohort_date"]).dt.days // 7)
retention = (
    activity.groupby(["cohort_date", "week_num"], as_index=False)
            .agg(retained_users=("user_id", "nunique"))
)

# Funnel analysis skeleton
funnel = pd.DataFrame({
    "step":  ["view", "cart", "checkout"],
    "users": [
        events.loc[events["step"] == "view",     "user_id"].nunique(),
        events.loc[events["step"] == "cart",     "user_id"].nunique(),
        events.loc[events["step"] == "checkout", "user_id"].nunique()
    ]
})
funnel["conversion"] = funnel["users"] / funnel["users"].iloc[0]

# Gaps & Islands — consecutive streaks
logins = logins.sort_values(["user_id", "login_date"])
logins["row_num"] = logins.groupby("user_id").cumcount() + 1
logins["grp"] = logins["login_date"] - pd.to_timedelta(logins["row_num"], unit="D")
streaks = (
    logins.groupby(["user_id", "grp"], as_index=False)
          .agg(streak_start=("login_date", "min"),
               streak_end  =("login_date", "max"),
               streak_len  =("login_date", "count"))
)

**Common mistakes:**
- Using `.dt.days` on a `DateOffset` result — use `.dt.days` only on `Timedelta`; `DateOffset` subtraction returns `Timedelta` automatically
- Gaps & Islands: forgetting to sort before `cumcount()` — produces incorrect group assignments
- Funnel: comparing `nunique()` across different filtered DataFrames — ensure all filters are applied to the same base DataFrame for consistency

---
## §10 — Performance Tips

In [ ]:
# Prefer vectorized operations over apply
df["col"] * 2                           # fast — vectorized
df["col"].apply(lambda x: x * 2)        # slow — row-by-row Python loop

# query vs boolean indexing
df.query("age > 30 and region == 'US'") # readable, slightly faster on large DataFrames
df[(df["age"] > 30) & (df["region"] == "US")]  # equivalent, more flexible for dynamic conditions

# Reduce memory with category dtype
df["platform"] = df["platform"].astype("category")  # useful for low-cardinality string columns

# Read only needed columns
pd.read_csv("file.csv", usecols=["user_id", "date", "revenue"])

# Process large files in chunks
chunks = pd.read_csv("large.csv", chunksize=100_000)
result = pd.concat([process(chunk) for chunk in chunks])

- Avoid `apply` with `axis=1` (row-wise) — it is a Python loop; use `np.where`, `np.select`, or vectorized column operations instead
- Avoid chained indexing (`df["col"][mask]`) — use `.loc[mask, "col"]` to avoid `SettingWithCopyWarning`
- `inplace=True` does not speed things up and makes code harder to chain — avoid it
- Use `.copy()` when slicing a DataFrame you intend to modify — prevents unintended modifications to the original

---
# Decision Guide

```
Aggregating data?
├── One metric, one grouping dimension      → groupby + agg
├── One metric across two categorical dims  → pivot_table
├── Frequency count only                   → crosstab
└── Need result back on every original row → groupby + transform

Filtering rows?
├── Static condition, readable              → .query()
└── Dynamic / programmatic condition        → boolean indexing df[mask]

Combining DataFrames?
├── Join on a column                        → .merge(on="col")
├── Join on index                           → .join() or .merge(left_index=True)
└── Stack rows vertically                   → pd.concat([df1, df2])

Window operations?
├── Rank within group                       → groupby().rank(method="dense")
├── Previous / next row value               → groupby().shift(1) / shift(-1)
├── Running total                           → groupby().cumsum()
└── Moving average over last n rows         → groupby().transform(lambda x: x.rolling(n).mean())

Labeling / bucketing values?
├── Binary condition                        → np.where(cond, true_val, false_val)
├── Multiple conditions                     → np.select(conditions, choices, default)
└── Binning continuous values               → pd.cut() or pd.qcut()

Speeding up slow code?
├── Using apply row-wise                    → replace with np.where / vectorized ops
├── Large file loading slowly               → usecols= or chunksize=
└── High-cardinality string columns         → .astype("category")
```